CASO DI STUDIO 1

In [1]:
import csv
import time

def genera_csv_scenario1_massivo(nome_file, numero_totale_nodi=1000000):
    print(f"⚙️ Inizio generazione dataset massivo: {nome_file}")
    print(f"📊 Nodi totali: {numero_totale_nodi}")
    
    inizio = time.time()
    
    # Apriamo il file in modalità scrittura ('w')
    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        
        # Scriviamo l'intestazione
        writer.writerow(['Source', 'Target'])
        
        # I nodi 0, 1, 2 sono gli influencer (non fanno nulla)
        # I follower partono dal nodo 3 fino a 999.999
        # Per evitare di stampare 1 milione di righe singole (lento), le prepariamo in blocchi
        
        blocco_dati = []
        dimensione_blocco = 50000 # Scrive sul file ogni 50.000 follower
        
        # Ogni follower segue i 3 influencer, quindi aggiungiamo 3 righe per ogni follower
        #gli influencer non seguono nessuno, quindi partiamo da 3, non hanno archi in uscita
        #i follower non si seguono tra di loro, quindi non hanno archi in uscita verso altri follower
        for follower in range(3, numero_totale_nodi):
            # Il follower segue i 3 influencer
            blocco_dati.append([follower, 0])
            blocco_dati.append([follower, 1])
            blocco_dati.append([follower, 2])
            
            # Quando il blocco è pieno, lo scriviamo sul disco e lo svuotiamo
            if len(blocco_dati) >= dimensione_blocco * 3:
                writer.writerows(blocco_dati)
                blocco_dati.clear()
                
        # Scriviamo gli eventuali dati rimasti nel blocco alla fine del ciclo
        if blocco_dati:
            writer.writerows(blocco_dati)
            
    fine = time.time()
    print(f"✅ Generazione completata con successo in {fine - inizio:.2f} secondi!")
    print(f"Il file '{nome_file}' è pronto nella tua cartella.")

# ==========================================
# ESECUZIONE
# ==========================================
# Puoi cambiare il numero di nodi se vuoi testarne 100.000, 500.000 o 1.000.000
N_NODI = 1000000
# Sostituisci la vecchia riga NOME_FILE con questa:
NOME_FILE = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv'

genera_csv_scenario1_massivo(NOME_FILE, N_NODI)

⚙️ Inizio generazione dataset massivo: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv
📊 Nodi totali: 1000000
✅ Generazione completata con successo in 0.76 secondi!
Il file '../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv' è pronto nella tua cartella.


In [3]:
import csv
import random
import time

def genera_csv_scenario2_massivo(nome_file, n_nodi=1000000):
    print(f"⚙️ Inizio generazione dataset massivo: {nome_file}")
    inizio = time.time()

    # Inizializziamo il seed per la riproducibilità dei risultati
    random.seed(42)

    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])

        blocco_dati = []
        dimensione_blocco = 100000 # Scrittura su disco ogni 100.000 righe

        def scrivi_blocco():
            """Funzione helper per svuotare la RAM scrivendo i dati sul CSV."""
            nonlocal blocco_dati
            if blocco_dati:
                writer.writerows(blocco_dati)
                blocco_dati.clear()

        # ==========================================
        # PASSO 1: INFLUENCER (0, 1, 2)
        # ==========================================
        # Circolo chiuso : 0->1, 1->2, 2->0
        #in questo modo gli influencer si seguono tra di loro, creando un piccolo circolo chiuso di 3 nodi
        blocco_dati.extend([[0, 1], [1, 2], [2, 0]])

        # ==========================================
        # PASSO 2: BOT INATTIVI (3 -> 499.999)
        # ==========================================
        print("📊 Generazione dei Bot inattivi in corso...")
        for bot in range(3, 500000):
            # Ogni bot segue i 3 influencer, ma non segue altri bot o utenti attivi
            blocco_dati.extend([[bot, 0], [bot, 1], [bot, 2]])
            
            # Svuota la RAM se il blocco è pieno
            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # ==========================================
        # PASSO 3: UTENTI ATTIVI E HUB (500.000 -> 999.999)
        # ==========================================
        print("🌐 Generazione degli Utenti Attivi e Hub in corso...")
        start_attivi = 500000
        end_attivi = 999999

        for utente in range(start_attivi, end_attivi + 1):
            # 1. Tutti gli utenti attivi seguono i 3 influencer
            blocco_dati.extend([[utente, 0], [utente, 1], [utente, 2]])

            # 2. Logica Hub Strategici
            # Eleggiamo a "Hub" un nodo ogni 500 (circa 1000 Hub totali)
            #cioè i nodi 500.000, 500.500, 501.000, ... saranno Hub con molti più link in uscita
            #ogni hub avrà 25 link in uscita verso altri utenti attivi, mentre gli utenti normali ne avranno solo 2
            is_hub = (utente % 500 == 0)
            
            # Gli Hub generano 25 link in uscita, gli utenti normali solo 2
            num_interazioni = 25 if is_hub else 2

            # Generazione delle interazioni tra utenti attivi
            for _ in range(num_interazioni):
                target_casuale = random.randint(start_attivi, end_attivi) # Target casuale tra gli utenti attivi
                if target_casuale != utente: # Evita l'auto-follow # (un utente non può seguire se stesso)
                    blocco_dati.append([utente, target_casuale]) # Aggiunge il link di follow

            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # Svuota gli ultimi dati rimasti in memoria alla fine del ciclo
        scrivi_blocco()

    fine = time.time()
    print(f"✅ Generazione completata con successo in {fine - inizio:.2f} secondi!")
    print(f"Il file '{nome_file}' è pronto.")

# ==========================================
# ESECUZIONE
# ==========================================
# Specifica il percorso corretto per la tua cartella
NOME_FILE = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv'

genera_csv_scenario2_massivo(NOME_FILE)

⚙️ Inizio generazione dataset massivo: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv
📊 Generazione dei Bot inattivi in corso...
🌐 Generazione degli Utenti Attivi e Hub in corso...
✅ Generazione completata con successo in 1.20 secondi!
Il file '../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv' è pronto.


In [ ]:
import csv
import random
import time

def genera_csv_scenario_3_massivo(nome_file, n_nodi=1000000):
    print(f"⚙️ Inizio generazione dataset massivo (Ottimizzato): {nome_file}")
    inizio = time.time()
    
    # Seed per garantire la riproducibilità scientifica
    random.seed(42)

    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])

        #ottimizzazione della scrittura su disco: accumuliamo i dati in memoria e li scriviamo a blocchi
        blocco_dati = []
        dimensione_blocco = 100000 # Salva su disco e svuota la RAM ogni 100k archi

        def scrivi_blocco():
            nonlocal blocco_dati
            if blocco_dati:
                writer.writerows(blocco_dati)
                blocco_dati.clear()

        #definizione degli influencer (0, 1, 2)
        influencers = [0, 1, 2] 
        
        # ===================================================
        # DEFINIZIONE DEI "FOLLOWER PIU' ATTIVI" (micro-hubs)
        # ===================================================
        # Selezioniamo un cluster di nodi (es. multipli di 1000).
        #quindi ci saranno 1000 nodi attivi (1000, 2000, 3000, ... fino a 999.000).
        # A questi nodi daremo una struttura comportamentale iper-attiva nel Passo 2.
        # I nodi 1000, 2000, 3000, ... saranno i "micro-hubs" con molti più link in uscita rispetto agli altri follower attivi
        micro_hubs = set(range(1000, n_nodi, 1000)) 
        micro_hubs_list = list(micro_hubs)
        
        # ===============================================
        # PASSO 1: RECIPROCITA' (Influencer -> Micro-Hub)
        # ==============================================
        print("🎯 Generazione archi di reciprocità (Influencer -> Nodi realmente attivi)...")
        for inf in influencers:
            # Ogni top creator segue un set di 150 utenti scelti ESCLUSIVAMENTE tra i follower più attivi
            # per simulare la reciprocità, ogni influencer segue 150 micro-hub (nodi 1000, 2000, 3000, ...) pescandoli casualmente
            hub_selezionati = random.sample(micro_hubs_list, 150)
            for hub in hub_selezionati:
                blocco_dati.append([inf, hub]) # Aggiunge il link di follow dall'influencer al micro-hub

        # ===============================================
        # PASSO 2: RETE FOLLOWER E DINAMICHE ORGANICHE
        # ==============================================
        print("🌐 Generazione network organico (Probabilità, Cluster e Hub reali)...")
        
        #iteramo su tutti i follower (da 3 a 999.999) per generare le loro connessioni
        for i in range(3, n_nodi):
            
            # 1. Preferential Attachment Asimmetrico 
            #random.random() genera un numero casuale tra 0 e 1.
            # Se è inferiore a 0.90, aggiungiamo un link verso l'influencer 0, e così via per gli altri influencer con probabilità decrescenti.
            if random.random() < 0.90: blocco_dati.append([i, 0]) # Il 90% dei follower segue l'influencer 0 (il più popolare)
            if random.random() < 0.70: blocco_dati.append([i, 1]) # Il 70% dei follower segue l'influencer 1
            if random.random() < 0.50: blocco_dati.append([i, 2]) # Il 50% dei follower segue l'influencer 2
            
            # 2. Generazione dell'Attività Locale (Hub vs Utenti Normali)
            # Se il nodo è un micro-hub, genera molti archi in uscita (fino a 15), altrimenti solo 1 o 2 link verso altri follower.
            if i in micro_hubs:
                # Questo nodo è un utente estremamente attivo. Crea fitte interazioni locali.
                # Ogni micro-hub segue i 15 follower più vicini (i successivi 15 nodi), simulando interazioni locali intense e clusterizzate.
                for j in range(1, 15):
                    if i + j < n_nodi:
                        blocco_dati.append([i, i + j]) # Gli hub seguono i follower più vicini (simulando interazioni locali intense)
            else:
                # Questo è un utente normale. Attivo, ma meno connesso (segue il vicino).
                if i + 1 < n_nodi:
                    blocco_dati.append([i, i + 1]) # Gli utenti normali seguono solo il nodo immediatamente successivo (simulando una connessione più debole e meno clusterizzata)
                
            # 3. Chiusure Triadiche (Small-World)
            # qui 1 utente su  3 decide di seguire qualcuno che è 2 posizioni indietro (i-2), creando così un legame di chiusura triadica ogni 3 nodi, che aiuta a formare cluster e comunità più coese.
            if i % 3 == 0 and (i - 2) >= 3:
                blocco_dati.append([i, i - 2])
                
            # 4. Bridging mirato (Attirato dai Micro-Hub)
            # Il 5% degli utenti compie un'azione di networking casuale
            # Questa azione ha l'effetto di attrarre nuovi follower verso i micro-hub, 
            # simulando l'effetto di attrazione gravitazionale che i nodi più attivi esercitano su quelli meno connessi.
            if random.random() < 0.05:
                # Per simulare l'attrazione gravitazionale, l'80% di queste azioni 
                # viene attratta dai nodi iper-attivi (micro-hubs) calcolati in precedenza.
                if random.random() < 0.80:
                    target_casuale = random.choice(micro_hubs_list)
                else:
                    target_casuale = random.randint(3, n_nodi - 1) # Target casuale tra tutti i follower (compresi i micro-hubs)
                
                # Evita l'auto-follow (un utente non può seguire se stesso)
                if target_casuale != i: 
                    blocco_dati.append([i, target_casuale]) # Aggiunge il link di follow casuale
            
            # Salvataggio chunk per non saturare la memoria
            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        scrivi_blocco()

    fine = time.time()
    print(f"✅ Generazione Scenario Massivo completata con successo in {fine - inizio:.2f} secondi!")

# Esecuzione del file
NOME_FILE = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv'
genera_csv_scenario_3_massivo(NOME_FILE)

⚙️ Inizio generazione dataset massivo (Ottimizzato): ../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv
🎯 Generazione archi di reciprocità (Influencer -> Nodi realmente attivi)...
🌐 Generazione network organico (Probabilità, Cluster e Hub reali)...
✅ Generazione Scenario Massivo completata con successo in 1.13 secondi!
